# Sistema de Processamento Visual — Contador de Comprimidos
**Equipe:** Os Laplacianos  
**Integrantes:** Matheus Foresto Moselli · Marcos Vinicius Medeiros da Silva · Karl Eloy Marques Henrique  
**Disciplina:** MCZA018 – Processamento Digital de Imagens  
**Data de Publicação:** 30/03/2026

In [1]:
# Equipe: Os Laplacianos
# Integrantes: Matheus Foresto Moselli, Marcos Vinicius Medeiros da Silva, Karl Eloy Marques Henrique
# Data: 30/03/2026  |  Programa: spv_contador_comprimidos.ipynb
# Descrição: SPV para contagem automática de comprimidos via processamento de imagem.
# Execução: jupyter notebook spv_contador_comprimidos.ipynb

import cv2
import numpy as np

## 2. Parâmetros Globais

In [2]:
# Pré-processamento
KERNEL_BLUR        = 31     # Tamanho do filtro Gaussiano
CLAHE_CLIP         = 2.0    # Limite de contraste do CLAHE
CLAHE_GRID         = (8, 8) # Grade de tiles do CLAHE

# Morfologia
KERNEL_MORPH       = 21     # Mantido igual ao código original

# Filtragem de regiões
AREA_MIN           = 1500   # Área mínima para considerar como comprimido (px²)
CIRCULARIDADE_MIN  = 0.6    # Circularidade mínima (0=qualquer, 1=círculo perfeito)

# Watershed
KERNEL_WATERSHED = 3
DIST_TRANSFORMATION_MASK_SIZE = 0
PERCENT_MAX_DISTANCE = 0.1

## 3. Fluxo de processamento

### [A] Pré-processamento
*Conceitos aplicados: **Processamento de Cores** e **Histograma / Equalização***

In [3]:
def preprocessar(img):
    """
    Aplica processamento de cores e equalização de histograma.

    Etapas:
        1. BGR → LAB  : separa luminância (L) de crominância (a, b),
                        tornando o contraste independente da cor.
        2. CLAHE      : equalização adaptativa com limite de contraste no
                        canal L — realça bordas de comprimidos sem
                        amplificar ruídos de fundo.
        3. Gaussiano  : suaviza ruídos de textura antes da binarização.

    Parâmetros:
        img : imagem BGR lida pelo OpenCV

    Retorna:
        blur     : imagem em tons de cinza suavizada (entrada para binarizar)
        img_rgb  : imagem original em RGB (para exibição via matplotlib)
    """
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Obtendo o canal L
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    canal_l = lab[:, :, 0]

    # Aplicando CLAHE (Contrast Limited Adaptive Histogram Equalization) no canal L
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    canal_l_eq = clahe.apply(canal_l)
    lab[:, :, 0] = canal_l_eq

    # Converter de volta para BGR e extrair cinza equalizado
    img_eq = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    gray   = cv2.cvtColor(img_eq, cv2.COLOR_BGR2GRAY)

    # Filtragem: Gaussiano
    blur = cv2.GaussianBlur(gray, (KERNEL_BLUR, KERNEL_BLUR), 0)

    return blur, img_rgb

### [B] Binarização e Morfologia
*Conceitos aplicados: **Operadores Morfológicos** e binarização por limiar de Otsu*

In [4]:
def binarizar(blur):
    """
    Binariza a imagem e aplica operadores morfológicos.

    Etapas:
        1. Otsu      : limiar automático — separa comprimidos do fundo
                       sem parâmetro manual, adaptando-se a diferentes
                       iluminações.
        2. MORPH_OPEN: erosão seguida de dilatação com kernel 15×15 —
                       remove ruídos menores que o kernel e suaviza
                       bordas dos comprimidos.

    Parâmetros:
        blur : imagem em tons de cinza suavizada (saída de preprocessar)

    Retorna:
        opening : máscara binária com regiões de comprimidos
    """
    # Binarização Otsu
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Operadores Morfológicos
    kernel  = np.ones((KERNEL_MORPH, KERNEL_MORPH), np.uint8)

    # Remove mancas brancas pequenas
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Remove buracos
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)

    return closing

### [C] Segmentação com Watershed
*Conceito aplicado: **Segmentação — Watershed***

In [5]:
def aplicar_watershed(img_binarizada, img_bgr):
    """
    Aplica o algoritmo Watershed para separar comprimidos sobrepostos.

    Etapas:
        1. Transformada de distância → picos = centros dos comprimidos
        2. Threshold nos picos → marcadores "frente certa" (objeto)
        3. Dilatação da máscara → marcadores "fundo certo"
        4. Região desconhecida = fundo_certo − frente_certa
        5. cv2.watershed() propaga os marcadores e traça bordas (−1)

    Parâmetros:
        img_binarizada  : máscara binária pós-morfologia (saída de binarizar)
        img_bgr  : imagem BGR original (necessária para cv2.watershed)

    Retorna:
        markers  : mapa de rótulos; cada comprimido recebe um ID único;
                   bordas Watershed marcadas com −1
    """
    kernel     = np.ones((KERNEL_WATERSHED, KERNEL_WATERSHED), np.uint8)
    fundo_certo = cv2.dilate(img_binarizada, kernel, iterations=3)

    dist_transform = cv2.distanceTransform(
        img_binarizada,
        cv2.DIST_L2,
        DIST_TRANSFORMATION_MASK_SIZE
    )

    _, frente_certa = cv2.threshold(
        dist_transform,
        PERCENT_MAX_DISTANCE * dist_transform.max(),
        255,
        0
    )

    frente_certa = np.uint8(frente_certa)

    desconhecido = cv2.subtract(fundo_certo, frente_certa)

    _, markers = cv2.connectedComponents(frente_certa)
    markers = markers + 1
    markers[desconhecido == 255] = 0

    markers = cv2.watershed(img_bgr, markers)
    # markers == -1 → borda entre dois objetos
    # markers == 1  → fundo
    # markers >= 2  → cada comprimido

    return markers

### [D] Detecção, Filtragem e Classificação

In [6]:
def detectar(markers, img_rgb, nome_arquivo=""):
    """
    Extrai contornos do mapa de rótulos do Watershed e filtra por
    área e circularidade — mantendo a lógica original do colega.

    Classificação:
        circularidade > 0.85 → "Redonda"
        circularidade > 0.60 → "Elipse"

    Parâmetros:
        markers      : mapa de rótulos do Watershed
        img_rgb      : imagem RGB para desenhar os resultados
        nome_arquivo : usado no título do plot

    Retorna:
        result : imagem RGB anotada com contornos e rótulos
        count  : número de comprimidos detectados
    """
    result = img_rgb.copy()
    count  = 0

    for region_id in np.unique(markers):
        if region_id <= 1:   # 1 = fundo, -1 = borda Watershed
            continue

        # Máscara isolada para esse comprimido
        mask_region = np.uint8(markers == region_id) * 255
        contours, _ = cv2.findContours(
            mask_region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            continue
        cnt = contours[0]

        area = cv2.contourArea(cnt)
        if area <= AREA_MIN:
            continue

        perimeter    = cv2.arcLength(cnt, True)
        circularity  = (4 * np.pi * area / (perimeter ** 2)
                        if perimeter > 0 else 0)

        if circularity <= CIRCULARIDADE_MIN:
            continue

        count += 1
        tipo   = "Redonda" if circularity > 0.85 else "Elipse"

        cv2.drawContours(result, [cnt], -1, (0, 255, 0), 5)

        M  = cv2.moments(cnt)
        cX = int(M["m10"] / M["m00"]) if M["m00"] != 0 else 0
        cY = int(M["m01"] / M["m00"]) if M["m00"] != 0 else 0

        label = f"#{count}: {tipo}"
        cv2.putText(
            result,
            label,
            (cX + 50, cY),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 255, 0),
            3
        )

    return result, count

## 4. Processamento da webcam
Visualiza a imagem da webcam e salva o video na pasta records.

In [7]:
def normalizar_para_hconcat(img, altura_ref):
    """Garante que a imagem está em BGR uint8 com a altura correta."""
    # 1. Converte dtype
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # 2. Garante 3 canais BGR
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

    # 3. Ajusta altura se necessário
    if img.shape[0] != altura_ref:
        img = cv2.resize(img, (img.shape[1], altura_ref))

    return img

In [8]:
from datetime import datetime
cap = cv2.VideoCapture(0)
cores_watershed = {}

if not cap.isOpened():
    print("Erro: webcam não encontrada!")
    exit()

ret, frame_test = cap.read()
if not ret:
    print("Erro ao capturar frame de teste")
    cap.release()
    exit()

h_ref = frame_test.shape[0]
p_test = normalizar_para_hconcat(frame_test, h_ref)
total_width_example = cv2.hconcat([p_test] * 5)
pipeline_h, pipeline_w = total_width_example.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'XVID')

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"records/{timestamp}.avi"

out = cv2.VideoWriter(
    filename,
    fourcc,
    30.0,
    (pipeline_w, pipeline_h)
)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Erro ao capturar frame")
        break

    # Aplicando etapas de tratamento
    blur, img_rgb = preprocessar(frame)
    closing = binarizar(blur)
    markers = aplicar_watershed(closing, frame.copy())
    result, count = detectar(markers, img_rgb)

    # Exibição das etapas
    # Mapa de rótulos colorido para visualizar o Watershed
    watershed_vis = np.zeros((*markers.shape, KERNEL_WATERSHED), dtype=np.uint8)
    for rid in np.unique(markers):
        if rid <= 1:
            continue

        if rid not in cores_watershed:
            cores_watershed[rid] = tuple(
                int(c) for c in np.random.randint(80, 255, 3, dtype=np.uint8)
            )

        watershed_vis[markers == rid] = cores_watershed[rid]

    def add_title(img, titulo):
        out = img.copy()

        cv2.putText(out,
                    titulo,
                    (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 0),
                    1,
                    cv2.LINE_AA
        )

        return out

    # Bordas Watershed em vermelho
    watershed_vis[markers == -1] = (255, 0, 0)

    p1 = add_title(frame,        "1. Original")
    p2 = add_title(blur,     "2. CLAHE+Gaussiano")
    p3 = add_title(closing,  "3. Mascara Otsu+OPEN")
    p4 = add_title(watershed_vis,"4. Watershed")
    p5 = add_title(result,       f"5. Resultado: {count}")

    p1 = normalizar_para_hconcat(p1, h_ref)
    p2 = normalizar_para_hconcat(p2, h_ref)
    p3 = normalizar_para_hconcat(p3, h_ref)
    p4 = normalizar_para_hconcat(p4, h_ref)
    p5 = normalizar_para_hconcat(p5, h_ref)

    pipeline = cv2.hconcat([p1, p2, p3, p4, p5])

    out.write(pipeline)

    # Mostra janela com resultado
    cv2.imshow("Contador de Comprimidos", pipeline)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):       # 'q' encerra
        break
    elif key == ord('s'):     # 's' salva o frame atual
        cv2.imwrite("captura.jpg", frame)
        print(f"Frame salvo!")

cap.release()
out.release()
cv2.destroyAllWindows()